# PMET Knowledge Editing — MUSE Dataset
**Goal:** Use PMET (Precise Model Editing in a Transformer) to suppress memorised knowledge on the MUSE-News benchmark.

**Pipeline:**
1. Load MUSE-News (VerbMem + KnowMem forget splits) as edit requests
2. Calibrate λ on a wikitext covariance
3. Run PMET batched editing (Stage-1 joint δ_a+δ_m optimisation + Stage-2 √-spread weight update)
4. Evaluate with official MUSE metrics (VerbMem ROUGE-L, KnowMem ROUGE-1, Perplexity, Forget Quality)

**Paper:** https://arxiv.org/abs/2308.08742  
**MUSE:** https://muse-bench.github.io (Shi et al. 2024)


In [2]:
import os, json, time, gc, re, math, warnings, itertools
from copy import deepcopy
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [3]:
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    login(tok); os.environ['HUGGING_FACE_HUB_TOKEN'] = tok
    print("HF login ✓")
except Exception as e:
    print(f"Skipping HF login: {e}")


HF login ✓


In [4]:
# ============================================================================
# CONFIG
# ============================================================================

@dataclass
class PMETConfig:
    model_name         : str       = 'meta-llama/Llama-3.2-1B'
    layers             : List[int] = field(default_factory=lambda: [3,4,5,6,7,8,9])
    mlp_module_tmp     : str       = 'model.layers.{}.mlp.down_proj'  # keys
    ffn_block_tmp      : str       = 'model.layers.{}.mlp'            # values
    layer_module_tmp   : str       = 'model.layers.{}'

    # Stage-1
    m_num_grad_steps   : int       = 100
    m_lr               : float     = 0.02
    m_weight_decay     : float     = 0.0
    kl_factor          : float     = 0.0625
    ce_factor          : float     = 1.0
    clamp_norm_factor  : float     = 4.0
    m_num_prefixes     : int       = 8

    # Covariance
    mom2_update_weight : float     = 15.0      # λ — auto-calibrated below
    cov_n_texts        : int       = 6000
    cov_ridge_eps      : float     = 1e-2

    batch_size         : int       = 10

CFG      = PMETConfig()
N_EDITS  = 10   # how many MUSE records to edit
EVAL_N   = 10   # how many to evaluate

PREFIX_BANK = [
    "", "As we know, ", "It is well established that ",
    "According to available information, ", "Historically speaking, ",
    "In fact, ", "It has been documented that ", "Based on records, ",
    "To be precise, ", "It is commonly known that ",
]

print('Config ✓')


Config ✓


In [5]:
# ============================================================================
# MODEL LOADING
# ============================================================================

tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name, torch_dtype=torch.float16, device_map={'': 0})
model.eval()

GPU0 = next(iter({p.device for p in model.parameters()}))
print(f'Model on {GPU0}  |  VRAM={torch.cuda.memory_allocated()/1e9:.2f} GB')


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model on cuda:0  |  VRAM=2.47 GB


In [6]:
# ============================================================================
# MUSE DATA — edit requests (for PMET Stage-1 + Stage-2)
# ============================================================================
# Separate from evaluation data. Keys: subject, target_new, prompt, ground_truth.
# VerbMem  → prompt = first half of article, target_new = next 20 words
#             (non-empty target required for Stage-1 CE loss to fire)
# KnowMem  → prompt = question, target_new = "I don't know"
#             (redirects the model's answer away from the true answer)

@dataclass
class EditRequest:
    subject     : str
    target_new  : str
    prompt      : str
    ground_truth: str

MUSE_EDIT_CACHE = '/kaggle/working/muse_edit_requests.json'

def build_muse_edit_requests(n: int = N_EDITS) -> List[EditRequest]:
    from datasets import load_dataset

    # Wipe any stale file that might have a different schema
    if os.path.exists(MUSE_EDIT_CACHE):
        os.remove(MUSE_EDIT_CACHE)

    raw = []

    # ── VerbMem: teach model to *not* reproduce the text ────────────────────
    try:
        vm = load_dataset('muse-bench/MUSE-News', 'verbmem', split='forget')
        for item in vm:
            text  = item.get('text', item.get('input', '')).strip()
            if not text:
                continue
            words = text.split()
            mid   = max(10, len(words) // 2)
            prompt_text = ' '.join(words[:mid])[:200]
            target_text = ' '.join(words[mid: mid+20])   # non-empty continuation
            caps = re.findall(r'(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)', prompt_text)
            subj = caps[0] if caps else prompt_text.split()[0]
            raw.append({'subset':'verbmem', 'prompt':prompt_text,
                        'target_new':target_text, 'subject':subj})
        print(f'  VerbMem: {sum(1 for r in raw if r["subset"]=="verbmem")} records')
    except Exception as e:
        print(f'  VerbMem load failed: {e}')

    # ── KnowMem: redirect QA answers to "I don't know" ──────────────────────
    try:
        km = load_dataset('muse-bench/MUSE-News', 'knowmem', split='forget_qa')
        for item in km:
            q   = item.get('question', item.get('input', '')).strip()
            ans = item.get('answer',   item.get('output', '')).strip()
            if not q:
                continue
            caps = re.findall(r'(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)', q)
            subj = caps[0] if caps else q.split()[0].rstrip('?.')
            raw.append({'subset':'knowmem', 'prompt':q,
                        'target_new':"I don't know", 'subject':subj,
                        'ground_truth': ans})
        print(f'  KnowMem: {sum(1 for r in raw if r["subset"]=="knowmem")} records')
    except Exception as e:
        print(f'  KnowMem load failed: {e}')

    with open(MUSE_EDIT_CACHE, 'w') as f:
        json.dump(raw, f)

    requests = [
        EditRequest(subject=r['subject'], target_new=r['target_new'],
                    prompt=r['prompt'], ground_truth=r.get('ground_truth',''))
        for r in raw[:n]
    ]
    return requests

muse_edit_requests = build_muse_edit_requests(N_EDITS)
print(f'\nTotal edit requests: {len(muse_edit_requests)}')
print('Sample:')
for r in muse_edit_requests[:3]:
    print(f"  subj='{r.subject[:25]}'  tgt='{r.target_new[:30]}'  prompt='{r.prompt[:50]}'")


README.md: 0.00B [00:00, ?B/s]

verbmem/forget-00000-of-00001.parquet:   0%|          | 0.00/295k [00:00<?, ?B/s]

Generating forget split:   0%|          | 0/100 [00:00<?, ? examples/s]

  VerbMem: 0 records


knowmem/retain_qa_icl-00000-of-00001.par(…):   0%|          | 0.00/2.90k [00:00<?, ?B/s]

...

Generating retain_qa_icl split:   0%|          | 0/10 [00:00<?, ? examples/s]

Generating retain_qa split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating forget_qa split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating forget_qa_icl split:   0%|          | 0/10 [00:00<?, ? examples/s]

  KnowMem: 100 records

Total edit requests: 10
Sample:
  subj='What'  tgt='I don't know'  prompt='What year did the Orkney Islands become part of Sc'
  subj='According'  tgt='I don't know'  prompt='According to Sir Elton, what year did Paul O'Grady'
  subj='In'  tgt='I don't know'  prompt='In what year did Willie Nelson start out as a song'


In [21]:
# ============================================================================
# MUSE DATA — raw eval records (separate cache, separate schema)
# ============================================================================
# Schema: {subset, text, question, answer}
# Must NOT share MUSE_EDIT_CACHE — different keys, different purpose.

MUSE_EVAL_CACHE = '/kaggle/working/muse_eval_raw.json'

def build_muse_eval_data() -> List[dict]:
    from datasets import load_dataset
    data = []

    try:
        vm = load_dataset('muse-bench/MUSE-News', 'verbmem', split='forget')
        for item in vm:
            text = item.get('prompt', '').strip()
            gt   = item.get('gt', '').strip()
            if text:
                data.append({'subset':'verbmem','text':text,'gt':gt,'question':'','answer':''})
        print(f'  VerbMem eval: {sum(1 for d in data if d["subset"]=="verbmem")} records')
    except Exception as e:
        print(f'  VerbMem eval load failed: {e}')

    try:
        km = load_dataset('muse-bench/MUSE-News', 'knowmem', split='forget_qa')
        for item in km:
            q   = item.get('question', item.get('input',  '')).strip()
            ans = item.get('answer',   item.get('output', '')).strip()
            if q and ans:
                data.append({'subset':'knowmem','text':'','question':q,'answer':ans})
        print(f'  KnowMem eval: {sum(1 for d in data if d["subset"]=="knowmem")} records')
    except Exception as e:
        print(f'  KnowMem eval load failed: {e}')

    with open(MUSE_EVAL_CACHE, 'w') as f:
        json.dump(data, f)
    print(f'  Total saved: {len(data)} records')
    return data

if os.path.exists(MUSE_EVAL_CACHE):
    with open(MUSE_EVAL_CACHE) as f:
        muse_eval = json.load(f)
    has_verbmem = any(d['subset'] == 'verbmem' and d['text'].strip() for d in muse_eval)
    if not muse_eval or 'text' not in muse_eval[0] or not has_verbmem:
        print('Stale eval cache — rebuilding...')
        os.remove(MUSE_EVAL_CACHE)
        muse_eval = build_muse_eval_data()
    else:
        print(f'Eval cache loaded: {len(muse_eval)} records')
else:
    muse_eval = build_muse_eval_data()

verbmem_eval = [d for d in muse_eval if d['subset']=='verbmem' and d['text'].strip()]
knowmem_eval = [d for d in muse_eval if d['subset']=='knowmem' and d['question'].strip()]
print(f'\nEval split — verbmem: {len(verbmem_eval)}  knowmem: {len(knowmem_eval)}')


  VerbMem eval: 100 records
  KnowMem eval: 100 records
  Total saved: 200 records

Eval split — verbmem: 100  knowmem: 100


In [8]:
# ============================================================================
# HELPERS
# ============================================================================

def get_module(model, name: str):
    m = model
    for p in name.split('.'): m = getattr(m, p)
    return m

def get_last_subject_pos(prompt: str, subject: str) -> int:
    """Token index of the last token of `subject` inside `prompt`.
    Falls back to last token if subject not found (safe for MUSE QA prompts).
    """
    prompt_ids = tokenizer(prompt, return_tensors='pt')['input_ids'][0]
    n = len(prompt_ids)
    for surface in [subject, ' ' + subject]:
        toks = tokenizer(surface, add_special_tokens=False)['input_ids']
        for i in range(n - len(toks), -1, -1):
            if prompt_ids[i: i+len(toks)].tolist() == toks:
                return i + len(toks) - 1
    # char-level fallback
    enc     = tokenizer(prompt, return_tensors='pt', return_offsets_mapping=True)
    offsets = enc['offset_mapping'][0]
    char_pos = prompt.lower().rfind(subject.lower())
    if char_pos != -1:
        last_char = char_pos + len(subject) - 1
        for tok_idx in range(len(offsets)-1, -1, -1):
            s, e = offsets[tok_idx].tolist()
            if s <= last_char <= e:
                return tok_idx
    return n - 1

print('Helpers ✓')


Helpers ✓


In [9]:
# ============================================================================
# COVARIANCE ESTIMATION  C = E[k k^T]  for down_proj input keys
# ============================================================================
# Uses wikitext-2 (diverse, independent of edit set).
# Hard-coded diverse fallback if wikitext unavailable — never uses edit_requests
# (that would bias C toward the target distribution → near-singular updates).

_cov_cache: Dict[int, torch.Tensor] = {}

_FALLBACK_TEXTS = [
    "The capital of France is Paris, a city known for the Eiffel Tower.",
    "Water boils at 100 degrees Celsius at sea level.",
    "Albert Einstein developed the theory of general relativity in 1915.",
    "The Amazon rainforest is the world's largest tropical rainforest.",
    "Shakespeare wrote Hamlet in the early 1600s.",
    "The speed of light in a vacuum is approximately 299,792 kilometres per second.",
    "DNA carries the genetic instructions for all living organisms.",
    "The Great Wall of China stretches over 21,000 kilometres.",
    "Isaac Newton formulated the laws of motion and universal gravitation.",
    "The Pacific Ocean is the largest and deepest ocean on Earth.",
    "The human brain contains approximately 86 billion neurons.",
    "Photosynthesis converts sunlight into chemical energy stored in glucose.",
    "The Pythagorean theorem states that a squared plus b squared equals c squared.",
    "Mount Everest is the tallest mountain above sea level on Earth.",
    "The Roman Empire fell in 476 AD when the last emperor was deposed.",
]

def estimate_cov_inv(layer_idx: int) -> torch.Tensor:
    if layer_idx in _cov_cache:
        return _cov_cache[layer_idx]
    print(f'  [Cov] Layer {layer_idx} ...', end=' ', flush=True)
    mod  = get_module(model, CFG.mlp_module_tmp.format(layer_idx))
    d_in = mod.weight.shape[1]

    buf = []
    def _hook(m, inp, out):
        buf.append(inp[0].detach().float().reshape(-1, d_in).cpu())
    h = mod.register_forward_hook(_hook)

    texts = None
    try:
        from datasets import load_dataset
        ds    = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train', streaming=True)
        texts = [ex['text'][:512] for ex, _ in zip(ds, range(CFG.cov_n_texts))
                 if len(ex['text'].strip()) > 40]
        print(f'[wikitext {len(texts)} seqs]', end=' ')
    except Exception as e:
        print(f'[wikitext fail: {e}]', end=' ')

    if not texts:
        texts = list(itertools.islice(itertools.cycle(_FALLBACK_TEXTS), CFG.cov_n_texts))
        print(f'[fallback {len(texts)} seqs]', end=' ')

    running = torch.zeros(d_in, d_in)
    total   = 0
    with torch.no_grad():
        for txt in texts:
            enc = tokenizer(txt, return_tensors='pt',
                            max_length=64, truncation=True).to(GPU0)
            model(**enc)
            if buf:
                k = buf.pop()
                running += k.T @ k
                total   += k.shape[0]
    h.remove()

    C   = running / max(total, 1)
    eps = CFG.cov_ridge_eps * C.trace().item() / d_in
    C_reg = C + eps * torch.eye(d_in, dtype=C.dtype)
    min_eig = torch.linalg.eigvalsh(C_reg).min().item()
    assert min_eig > 0, f'C not PD at layer {layer_idx}! min_eig={min_eig:.2e}'
    C_inv = torch.linalg.inv(C_reg)
    _cov_cache[layer_idx] = C_inv
    print(f'done  eps={eps:.2e}  min_eig={min_eig:.2e}  n={total}')
    return C_inv

print('Covariance ✓')


Covariance ✓


In [10]:
# ============================================================================
# AUTO-CALIBRATE λ  (mom2_update_weight)
# ============================================================================
# λ = 0.75 * median(k^T C^{-1} k)  over muse_edit_requests.
# Key position = get_last_subject_pos (falls back to last token for QA prompts).

_cov_cache.clear()
layer0 = CFG.layers[0]
C_inv0 = estimate_cov_inv(layer0).float().to(GPU0)

ktck_vals = []
for req in muse_edit_requests:
    s_pos   = get_last_subject_pos(req.prompt, req.subject)
    pmt_ids = tokenizer(req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
    cap = {}
    def _hk(m, inp, out, c=cap): c['k'] = inp[0].detach().float()
    hdl = get_module(model, CFG.mlp_module_tmp.format(layer0)).register_forward_hook(_hk)
    with torch.no_grad(): model(pmt_ids)
    hdl.remove()
    k = cap['k'][0, s_pos]
    ktck_vals.append((k @ C_inv0 @ k).item())

arr    = np.array(ktck_vals)
median = float(np.median(arr))
print(f'k^T C⁻¹k — min={arr.min():.1f}  median={median:.1f}  max={arr.max():.1f}')
assert median > 0, 'Calibration failed — check model and covariance'
CFG.mom2_update_weight = median * 0.75
_cov_cache.clear()
print(f'λ = {CFG.mom2_update_weight:.2f}')


  [Cov] Layer 3 ... 

README.md: 0.00B [00:00, ?B/s]

[wikitext 2761 seqs] done  eps=1.57e-05  min_eig=1.01e-04  n=154587
k^T C⁻¹k — min=6614.4  median=10335.4  max=10530.3
λ = 7751.57


In [11]:
# ============================================================================
# STAGE 1: Joint δ_a + δ_m optimisation  (PMET Eq.10)
# ============================================================================
# Optimises both δ_a (MHSA output perturbation) and δ_m (MLP block output
# perturbation) jointly at the anchor layer (CFG.layers[-1]).
# Both live in d_model space.

def compute_z_targets(requests: List[EditRequest],
                      prefixes: List[str]) -> torch.Tensor:
    """Returns m_targets [N, d_model] — optimised FFN-block output states."""
    anchor    = CFG.layers[-1]
    layer_mod = get_module(model, CFG.layer_module_tmp.format(anchor))
    ffn_mod   = get_module(model, CFG.ffn_block_tmp.format(anchor))

    m_target_list = []

    for idx, req in enumerate(requests):
        print(f'  [z] ({idx+1}/{len(requests)}) "{req.prompt[:40]}" → "{req.target_new[:25]}"')

        pmt_ids = tokenizer(req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
        tgt_ids = tokenizer(' ' + req.target_new, add_special_tokens=False,
                            return_tensors='pt')['input_ids'].to(GPU0)
        # Guard: if target_new tokenises to empty (edge case), skip
        if tgt_ids.shape[1] == 0:
            tgt_ids = tokenizer("unknown", add_special_tokens=False,
                                return_tensors='pt')['input_ids'].to(GPU0)

        full_ids_base = torch.cat([pmt_ids, tgt_ids], dim=1)
        s_pos = get_last_subject_pos(req.prompt, req.subject)

        # Capture baseline a^L (MHSA output) and m^L (MLP block output)
        cap = {}
        def _hook_a(m, inp, out, c=cap):
            c['a'] = (out[0] if isinstance(out, tuple) else out).detach().float()
        def _hook_m(m, inp, out, c=cap):
            c['m'] = (out[0] if isinstance(out, tuple) else out).detach().float()
        ha = layer_mod.self_attn.register_forward_hook(_hook_a)
        hm = ffn_mod.register_forward_hook(_hook_m)
        with torch.no_grad(): model(full_ids_base)
        ha.remove(); hm.remove()

        a_base = cap['a'][0, s_pos].clone()   # (d_model,)
        m_base = cap['m'][0, s_pos].clone()   # (d_model,)

        delta_a = torch.zeros_like(a_base, requires_grad=True)
        delta_m = torch.zeros_like(m_base, requires_grad=True)
        opt   = torch.optim.Adam([delta_a, delta_m],
                                  lr=CFG.m_lr, weight_decay=CFG.m_weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    opt, mode='min', factor=0.5, patience=8, min_lr=1e-4)

        # Cache reference logits (no-grad) for KL
        ref_cache = {}
        with torch.no_grad():
            for pv in prefixes:
                pv_ids = tokenizer(pv + req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
                fids   = torch.cat([pv_ids, tgt_ids], dim=1)
                ref_cache[pv] = model(fids).logits[0].float().detach()

        def patch_forward(pv: str):
            pv_ids   = tokenizer(pv + req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
            fids     = torch.cat([pv_ids, tgt_ids], dim=1)
            pv_len   = pv_ids.shape[1]
            plain_len = pmt_ids.shape[1]
            s_pos_v  = s_pos + (pv_len - plain_len)

            handles = []
            def _pa(m, inp, out):
                out_t = out[0] if isinstance(out, tuple) else out
                out_t = out_t.float()
                out_t[0, s_pos_v] = out_t[0, s_pos_v] + delta_a
                return (out_t.to(out[0].dtype),) + out[1:] if isinstance(out, tuple) else out_t.to(out.dtype)
            def _pm(m, inp, out):
                out_t = out[0] if isinstance(out, tuple) else out
                out_t = out_t.float()
                out_t[0, s_pos_v] = out_t[0, s_pos_v] + delta_m
                return (out_t.to(out[0].dtype),) + out[1:] if isinstance(out, tuple) else out_t.to(out.dtype)
            handles.append(layer_mod.self_attn.register_forward_hook(_pa))
            handles.append(ffn_mod.register_forward_hook(_pm))
            logits = model(fids).logits[0].float()
            for hh in handles: hh.remove()
            return logits, pv_len

        for step in range(CFG.m_num_grad_steps):
            opt.zero_grad()
            total_loss = torch.tensor(0.0, device=GPU0)

            for pv in prefixes:
                logits, pv_len = patch_forward(pv)
                # CE loss on target tokens
                tgt_start = pv_len - 1   # pv_ids already includes prompt; last prompt logit → pred starts here
                ce = F.cross_entropy(
                    logits[tgt_start: tgt_start + tgt_ids.shape[1]],
                    tgt_ids[0])
                # KL loss vs reference
                ref_lp  = F.log_softmax(ref_cache[pv], dim=-1)
                edit_lp = F.log_softmax(logits, dim=-1)
                kl = F.kl_div(edit_lp, ref_lp.exp(), reduction='batchmean')
                total_loss = total_loss + CFG.ce_factor * ce + CFG.kl_factor * kl

            total_loss = total_loss / len(prefixes)
            total_loss.backward()

            # Gradient clamp on delta_m
            if delta_m.grad is not None:
                max_norm = CFG.clamp_norm_factor * m_base.norm().item()
                g_norm   = delta_m.grad.norm().item()
                if g_norm > max_norm:
                    delta_m.grad.mul_(max_norm / (g_norm + 1e-8))

            opt.step()
            sched.step(total_loss)

            if step % 25 == 0:
                ce_val = (total_loss - CFG.kl_factor * kl).item()
                print(f'      step {step:3d} | CE={ce_val:.3f}  KL={kl.item():.4f}')

        m_target_list.append((m_base + delta_m).detach().cpu())
        del delta_a, delta_m, opt, sched, ref_cache
        gc.collect(); torch.cuda.empty_cache()

    return torch.stack(m_target_list, dim=0)   # (N, d_model)

print('Stage-1 ✓')


Stage-1 ✓


In [12]:
# ============================================================================
# STAGE 2: Closed-form weight update with √-spread  (PMET Eq.9 + Eq.11)
# ============================================================================
# Keys  k^l_i = inp[0] of down_proj  → shape (d_ffn_inner,)
# W0 (down_proj) : (d_model, d_ffn_inner)
# ΔW = R · (C⁻¹K)^T · (K^T C⁻¹ K + λI)^{-1}
# R  = (V1 - W0·K) / √(L_max - l + 1)

def collect_keys_for_layer(requests: List[EditRequest],
                            layer_idx: int,
                            prefixes: List[str]) -> torch.Tensor:
    """Returns K shape (d_ffn_inner, N), averaged over prefixes."""
    mod  = get_module(model, CFG.mlp_module_tmp.format(layer_idx))
    keys = []
    for req in requests:
        s_pos_plain = get_last_subject_pos(req.prompt, req.subject)
        plain_len   = tokenizer(req.prompt, return_tensors='pt')['input_ids'].shape[1]
        key_acc = torch.zeros(mod.weight.shape[1], dtype=torch.float32)
        for pv in prefixes:
            pv_ids    = tokenizer(pv + req.prompt, return_tensors='pt')['input_ids'].to(GPU0)
            prefix_len = pv_ids.shape[1] - plain_len
            s_pos_v   = s_pos_plain + prefix_len
            cap = {}
            def hook(m, inp, out, c=cap): c['k'] = inp[0].detach().float()
            hdl = mod.register_forward_hook(hook)
            with torch.no_grad(): model(pv_ids)
            hdl.remove()
            key_acc += cap['k'][0, s_pos_v].cpu()
        keys.append(key_acc / len(prefixes))
    return torch.stack(keys, dim=1)   # (d_ffn_inner, N)


def compute_delta_W(m_targets: torch.Tensor,
                    layer_idx: int,
                    L_max: int,
                    requests: List[EditRequest],
                    prefixes: List[str],
                    C_inv: torch.Tensor) -> torch.Tensor:
    mod = get_module(model, CFG.mlp_module_tmp.format(layer_idx))
    dev = mod.weight.device
    W0  = mod.weight.float()                                     # (d_model, d_ffn_inner)

    K    = collect_keys_for_layer(requests, layer_idx, prefixes).to(dev)  # (d_ffn_inner, N)
    W0K1 = W0 @ K                                                # (d_model, N)

    denom = math.sqrt(float(L_max - layer_idx + 1))
    V1    = m_targets.T.float().to(dev)                          # (d_model, N)
    R     = (V1 - W0K1) / denom                                  # (d_model, N)

    Ci    = C_inv.float().to(dev)                                # (d_ffn_inner, d_ffn_inner)
    CiK   = Ci @ K                                               # (d_ffn_inner, N)
    A     = K.T @ CiK + CFG.mom2_update_weight * torch.eye(K.shape[1], device=dev)
    dW    = R @ torch.linalg.solve(A.T, CiK.T)                  # (d_model, d_ffn_inner)

    # Norm clamp
    max_norm = CFG.clamp_norm_factor * W0.norm().item()
    if dW.norm().item() > max_norm:
        dW = dW * (max_norm / (dW.norm().item() + 1e-8))

    return dW

print('Stage-2 ✓')


Stage-2 ✓


In [13]:
# ============================================================================
# PMET APPLICATION
# ============================================================================

def apply_pmet(requests):
    print(f'\n=== PMET | edits={len(requests)} | layers={CFG.layers} | λ={CFG.mom2_update_weight:.1f} ===')
    prefixes = PREFIX_BANK[:CFG.m_num_prefixes]
    L_max    = CFG.layers[-1]

    print('\n[1/2] Stage-1: joint δ_a + δ_m optimisation ...')
    m_targets = compute_z_targets(requests, prefixes)
    print(f'  m_targets: {m_targets.shape}')

    print('\n[2/2] Stage-2: √-spread closed-form weight update ...')
    for layer in CFG.layers:
        C_inv = estimate_cov_inv(layer)
        dW    = compute_delta_W(m_targets, layer, L_max, requests, prefixes, C_inv)
        mod   = get_module(model, CFG.mlp_module_tmp.format(layer))
        with torch.no_grad():
            mod.weight.add_(dW.to(mod.weight.dtype))
        print(f'  Layer {layer}: ΔW={dW.shape}  ‖ΔW‖={dW.norm():.4f}')
        gc.collect(); torch.cuda.empty_cache()

    print('\nBatch complete ✓')


def apply_pmet_batched(requests):
    total     = len(requests)
    bs        = CFG.batch_size
    n_batches = (total + bs - 1) // bs
    print(f'\nRunning {total} edits in {n_batches} batches of ≤{bs}')
    for i in range(0, total, bs):
        batch = requests[i: i+bs]
        sep   = "=" * 60
        print(f'\n{sep}\nBatch {i//bs+1}/{n_batches}  (requests {i+1}–{min(i+bs,total)} of {total})\n{sep}')
        apply_pmet(batch)
    print(f'\n✓ All {total} edits complete.')

print('PMET runner ✓')


PMET runner ✓


In [14]:
# ============================================================================
# RUN PMET ON MUSE
# ============================================================================

t0 = time.time()
apply_pmet_batched(muse_edit_requests)
print(f'\nTotal editing time: {time.time()-t0:.1f}s')



Running 10 edits in 1 batches of ≤10

Batch 1/1  (requests 1–10 of 10)

=== PMET | edits=10 | layers=[3, 4, 5, 6, 7, 8, 9] | λ=7751.6 ===

[1/2] Stage-1: joint δ_a + δ_m optimisation ...
  [z] (1/10) "What year did the Orkney Islands become " → "I don't know"
      step   0 | CE=2.704  KL=0.0000
      step  25 | CE=0.030  KL=1.0177
      step  50 | CE=0.024  KL=0.7541
      step  75 | CE=0.020  KL=0.7200
  [z] (2/10) "According to Sir Elton, what year did Pa" → "I don't know"
      step   0 | CE=2.232  KL=-0.0000
      step  25 | CE=0.009  KL=0.8405
      step  50 | CE=0.008  KL=0.5903
      step  75 | CE=0.006  KL=0.5528
  [z] (3/10) "In what year did Willie Nelson start out" → "I don't know"
      step   0 | CE=2.389  KL=0.0000
      step  25 | CE=0.039  KL=1.4964
      step  50 | CE=0.013  KL=1.0330
      step  75 | CE=0.013  KL=0.9067
  [z] (4/10) "What percentage did the AfD party reach " → "I don't know"
      step   0 | CE=2.376  KL=0.0000
      step  25 | CE=0.023  KL=0.7882
 

In [22]:
# ============================================================================
# MUSE EVALUATION  (post-PMET)
# ============================================================================
# Metrics follow the official MUSE-Bench protocol (Shi et al. 2024):
#   VerbMem  ROUGE-L  (↓ better) — greedy continuation vs true text
#   KnowMem  ROUGE-1  (↓ better) — generated answer vs ground truth
#   Perplexity forget (↑ better) — model treats forget-set as unfamiliar
#   Forget Quality    (↑ better) — 1 - mean(VerbMem, KnowMem)

# ── ROUGE setup ──────────────────────────────────────────────────────────────
try:
    from rouge_score import rouge_scorer as rs_lib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rouge-score', '-q'])
    from rouge_score import rouge_scorer as rs_lib

_rscorer = rs_lib.RougeScorer(['rougeL', 'rouge1'], use_stemmer=False)

def rouge_l(ref, hyp):
    return _rscorer.score(ref, hyp)['rougeL'].fmeasure

def rouge_1(ref, hyp):
    return _rscorer.score(ref, hyp)['rouge1'].fmeasure

# ── VerbMem: greedy continuation ROUGE-L ─────────────────────────────────────
@torch.no_grad()
def verbmem_score(d):
    """ROUGE-L between model's greedy continuation and true gt. Lower = better."""
    prompt = d['text']   # already the prompt portion
    gt     = d['gt']     # already the ground truth continuation
    if not gt.strip():
        return 0.0
    enc = tokenizer(prompt, return_tensors='pt', max_length=256, truncation=True)
    input_ids      = enc['input_ids'].to(GPU0)
    attention_mask = enc['attention_mask'].to(GPU0)
    out = model.generate(input_ids, attention_mask=attention_mask,
                          max_new_tokens=100, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    return rouge_l(gt, gen)
# ── KnowMem: QA ROUGE-1 ──────────────────────────────────────────────────────
@torch.no_grad()
def knowmem_score(question, answer):
    """Lower = model can't answer. ↓ good."""
    enc            = tokenizer(question, return_tensors='pt', max_length=256, truncation=True)
    input_ids      = enc['input_ids'].to(GPU0)
    attention_mask = enc['attention_mask'].to(GPU0)
    out = model.generate(input_ids, attention_mask=attention_mask,
                          max_new_tokens=64, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][input_ids.shape[1]:],
                            skip_special_tokens=True).strip()
    return rouge_1(answer, gen)
# ── Perplexity on forget text ─────────────────────────────────────────────────
@torch.no_grad()
def perplexity(text, max_len=256):
    """Higher = model treats text as unfamiliar. ↑ good."""
    enc = tokenizer(text, return_tensors='pt', max_length=max_len, truncation=True).to(GPU0)
    ids = enc['input_ids']
    if ids.shape[1] < 2: return float('inf')
    logits = model(**enc).logits[0][:-1].float()
    lp  = torch.log_softmax(logits, dim=-1)
    tgt = ids[0, 1:]
    nll = -lp[range(len(tgt)), tgt].mean().item()
    return float(math.exp(nll))

# ── Run ───────────────────────────────────────────────────────────────────────
EVAL_N_VM = min(30, len(verbmem_eval))
EVAL_N_KM = min(30, len(knowmem_eval))

print(f'Running VerbMem ROUGE-L on {EVAL_N_VM} records...')
vm_scores = []
for i, d in enumerate(verbmem_eval[:EVAL_N_VM]):
    s = verbmem_score(d)
    vm_scores.append(s)
    if (i+1) % 5 == 0 or (i+1) == EVAL_N_VM:
        print(f'  [{i+1}/{EVAL_N_VM}] running avg = {np.mean(vm_scores):.4f}')

print(f'\nRunning KnowMem ROUGE-1 on {EVAL_N_KM} records...')
km_scores = []
for i, d in enumerate(knowmem_eval[:EVAL_N_KM]):
    s = knowmem_score(d['question'], d['answer'])
    km_scores.append(s)
    if (i+1) % 5 == 0 or (i+1) == EVAL_N_KM:
        print(f'  [{i+1}/{EVAL_N_KM}] running avg = {np.mean(km_scores):.4f}')

print(f'\nRunning Perplexity on {EVAL_N_VM} forget-set records...')
vm_ppx = [min(perplexity(d['text']), 1e6) for d in verbmem_eval[:EVAL_N_VM]]

# ── Aggregate ─────────────────────────────────────────────────────────────────
avg_verbmem    = float(np.mean(vm_scores)) if vm_scores else 0.0
avg_knowmem    = float(np.mean(km_scores)) if km_scores else 0.0
avg_ppx        = float(np.mean(vm_ppx))    if vm_ppx    else 0.0
ppx_norm       = min(math.log(avg_ppx + 1) / math.log(1e4 + 1), 1.0) if avg_ppx > 0 else 0.0
forget_quality = max(0.0, 1.0 - (avg_verbmem + avg_knowmem) / 2.0)

sep = "=" * 62
dash = "-" * 62
print(f'\n{sep}')
print('MUSE EVALUATION  (post-PMET edit)')
print(sep)
print(f'  {"Metric":<50} {"Value":>8}')
print(dash)
rows = [
    ('VerbMem ROUGE-L  (↓ better — less verbatim reproduction)', avg_verbmem),
    ('KnowMem ROUGE-1  (↓ better — less QA recall)',             avg_knowmem),
    ('Perplexity on forget set (avg)',                            avg_ppx),
    ('PPX normalised   (↑ better — scaled to [0,1])',            ppx_norm),
    ('Forget Quality   (↑ better — 1 - mean ROUGE)',             forget_quality),
]
for label, val in rows:
    print(f'  {label:<50} {val:>8.4f}')
print(sep)
print(f'\n  Evaluated on: {EVAL_N_VM} verbmem + {EVAL_N_KM} knowmem records')
print('\nMUSE evaluation complete ✓')


Running VerbMem ROUGE-L on 30 records...
  [5/30] running avg = 0.1111
  [10/30] running avg = 0.1221
  [15/30] running avg = 0.1167
  [20/30] running avg = 0.1191
  [25/30] running avg = 0.1222
  [30/30] running avg = 0.1216

Running KnowMem ROUGE-1 on 30 records...
  [5/30] running avg = 0.0000
  [10/30] running avg = 0.0000
  [15/30] running avg = 0.0000
  [20/30] running avg = 0.0020
  [25/30] running avg = 0.0016
  [30/30] running avg = 0.0026

Running Perplexity on 30 forget-set records...

MUSE EVALUATION  (post-PMET edit)
  Metric                                                Value
--------------------------------------------------------------
  VerbMem ROUGE-L  (↓ better — less verbatim reproduction)   0.1216
  KnowMem ROUGE-1  (↓ better — less QA recall)         0.0026
  Perplexity on forget set (avg)                      16.0820
  PPX normalised   (↑ better — scaled to [0,1])        0.3081
  Forget Quality   (↑ better — 1 - mean ROUGE)         0.9379

  Evaluated on: 30 ver

In [16]:
import os
for f in ['/kaggle/working/muse_eval_raw.json', '/kaggle/working/muse_news.json']:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")
print("Done")

Deleted /kaggle/working/muse_eval_raw.json
Done


In [17]:
from datasets import load_dataset
vm = load_dataset('muse-bench/MUSE-News', 'verbmem', split='forget')
print(f"Total rows: {len(vm)}")
print(f"Columns: {vm.column_names}")
print(f"First row keys/values:")
for k, v in vm[0].items():
    print(f"  {k!r}: {str(v)[:100]!r}")

Total rows: 100
Columns: ['prompt', 'gt']
First row keys/values:
  'prompt': 'now, everyone should just take a deep breath."\n\nAnalysts have said the US banking system - which has'
  'gt': 'down on his net worth in November 2021, when he became the first person ever to be worth $300bn (£23'


In [20]:
import os
os.remove('/kaggle/working/muse_eval_raw.json')
print("Cache cleared")

Cache cleared


In [23]:
for d in knowmem_eval[:5]:
    enc = tokenizer(d['question'], return_tensors='pt', max_length=256, truncation=True)
    input_ids      = enc['input_ids'].to(GPU0)
    attention_mask = enc['attention_mask'].to(GPU0)
    out = model.generate(input_ids, attention_mask=attention_mask,
                          max_new_tokens=64, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f'Q:    {d["question"]}')
    print(f'pred: {gen!r}')
    print(f'gold: {d["answer"]!r}')
    print()

Q:    What year did the Orkney Islands become part of Scotland?
pred: "I don't know.\nI don't know the date of the Orkney Islands becoming part of Scotland. I don't know the date of the Orkney Islands becoming part of Scotland.\nI don't know the date of the Orkney Islands becoming part of Scotland. I don't know the date of the"
gold: '1472'

Q:    According to Sir Elton, what year did Paul O'Grady host his and David Furnish's stag party?
pred: "I don't know.\nI don't know what the other person's name is, but I don't know his name.\nI don't know what the other person's name is, but I don't know his name.\nI don't know the other person's name, but I don't know his name.\nI"
gold: '2005'

Q:    In what year did Willie Nelson start out as a songwriter?
pred: "I don't know. I don't know how old he is. I don't know how old he is. I don't know how old he is. I don't know how old he is. I don't know how old he is. I don't know how old he is. I don't know how"
gold: '1960s'

Q:    What percenta